In [1]:
import os
import sys

repo_root_path = os.path.abspath(os.path.join("..", "..", ".."))

if repo_root_path not in sys.path:
    sys.path.append(repo_root_path)


In [ ]:
from analyse.advertising.e01_rupture_detection.script import (
    RuptureDetector,
    print_summary,
    export_json,
    plot_results,
)

INPUT_FILE = "/root/Workspace/quotaclimat/analyse/advertising/exports/tf1_2025-03-04_12-00-00_2025-03-04_13-00-00.mp3"
# INPUT_FILE = "/root/Workspace/quotaclimat/analyse/advertising/exports/small.mp3"
SENSITIVITY = 0.35
MIN_SEGMENT_DURATION = 2.0
OUT_JSON = "segments.json"
OUT_PLOT = "rupture_analyse.png"

In [ ]:
detector = RuptureDetector(
    sensitivity=SENSITIVITY,
    min_segment_sec=MIN_SEGMENT_DURATION,
)

segments, novelty, features, y = detector.run(INPUT_FILE)

print_summary(segments)
export_json(segments, OUT_JSON)

plot_results(
    y,
    detector.sr,
    novelty,
    segments,
    hop_length=detector.hop_length,
    output_path=OUT_PLOT,
)

In [15]:
from datetime import datetime

from analyse.advertising.tools.testimony_table.extract import get_testimony_data

start_time = datetime(2025, 3, 4, 12, 00, 0)

annotations = get_testimony_data(
    channel="TF1",
    from_date=start_time,
    to_date=datetime(2025, 3, 4, 13, 0, 0),
)

print(
    """
""".join(
        [
            "type,start,end",
            *(
                ",".join(
                    [
                        annotation["type"],
                        str(annotation["start"] - start_time),
                        str(annotation["end"] - start_time),
                    ]
                )
                for annotation in annotations
            ),
        ]
    )
)

print(
    [
        {
            "type": annotation["type"],
            "start_sec": (annotation["start"] - start_time).total_seconds(),
            "end_sec": (annotation["end"] - start_time).total_seconds(),
        }
        for annotation in annotations
    ]
)


type,start,end
PUBLICITE,0:25:37,0:31:43
PUBLICITE,0:45:46,0:47:18
PUBLICITE,0:52:53,0:57:33
[{'type': 'PUBLICITE', 'start_sec': 1537.0, 'end_sec': 1903.0}, {'type': 'PUBLICITE', 'start_sec': 2746.0, 'end_sec': 2838.0}, {'type': 'PUBLICITE', 'start_sec': 3173.0, 'end_sec': 3453.0}]


In [16]:
from datetime import datetime

from analyse.advertising.tools.segment_player.player_generator import generate_player
import json

from analyse.advertising.tools.testimony_table.extract import get_testimony_data

with open(OUT_JSON, encoding="utf-8") as f:
    segments = json.load(f)

annotations = [
    {"type": "PUBLICITE", "start_sec": 1537.0, "end_sec": 1903.0},
    {"type": "PUBLICITE", "start_sec": 2746.0, "end_sec": 2838.0},
    {"type": "PUBLICITE", "start_sec": 3173.0, "end_sec": 3453.0},
]

generate_player(
    media_input="https://vod.mediatree.fr/assets/tf1_2025-03-04T12-00-00Z_2025-03-04T13-00-00Z.mp4",
    segments=segments,
    annotations=annotations,
    output_path="player_output.html",
)


[1/3] URL média détectée : https://vod.mediatree.fr/assets/tf1_2025-03-04T12-00-00Z_2025-03-04T13-00-00Z.mp4
  Type     : vidéo
  Le fichier ne sera PAS embarqué dans le HTML.
[2/3] Segments fournis en entrée
  1103 segments
[3/3] Annotations fournies en entrée
  3 annotations

Génération du rapport HTML : player_output.html...

✓  Rapport généré  : player_output.html
   Taille HTML     : 0.4 Mo (léger, média chargé à la volée)
   URL média       : https://vod.mediatree.fr/assets/tf1_2025-03-04T12-00-00Z_2025-03-04T13-00-00Z.mp4
   Segments        : 1103
   Annotations CSV : 3

   → Ouvrez dans Firefox ou Chrome pour l'analyse.

